In [5]:
import requests

URL_ALL = "https://api.steampowered.com/ISteamApps/GetAppList/v2/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
}

def get_all_apps():
    # 取得全部 app 清單（每筆含 appid 與 name）。
    resp = requests.get(URL_ALL, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json().get("applist", {}).get("apps", [])

In [ ]:
import time

URL_DETAIL = "https://store.steampowered.com/api/appdetails"

def get_game_info_if_game(appid: int) -> dict | None:
    """
    全量 call appdetails（不帶 filters/cc/l）。
    若為 game，回傳只含指定欄位的 dict；否則回 None。
    """
    params = {"appids": str(appid)}
    tries = 1 + 2
    for _ in range(tries):
        try:
            r = requests.get(URL_DETAIL, params=params, headers=HEADERS, timeout=30)
            if r.status_code != 200:
                time.sleep(0.5); continue

            payload = r.json()
            root = payload.get(str(appid), {})
            if not root.get("success"):
                return None

            info = root.get("data") or {}
            if info.get("type") != "game":
                return None

            # developers 兼容舊欄位 'developer'
            developers = info.get("developers")
            if not developers:
                dev = info.get("developer")
                developers = [dev] if dev else []

            return {
                "name": info.get("name"),
                "steam_gameid": info.get("steam_appid"),
                "is_free": info.get("is_free", False),
                "supported_languages": info.get("supported_languages"),
                "developers": developers,
                "price_overview": info.get("price_overview"),  # 免費/區不可售可能沒有
                "platforms": info.get("platforms"),
                "genres": info.get("genres") or [],
            }

        except requests.RequestException:
            time.sleep(0.5)
    return None

get_game_info_if_game(appid=80)


In [3]:
# v0 極簡：單一 appid，成功路徑，取指定欄位後 print
import requests, os, pandas as pd  # 想純印就不用 pandas；我先放著，等下可一行存 CSV

URL = "https://store.steampowered.com/api/appdetails"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def tiny_game_info(appid: int):
    r = requests.get(URL, params={"appids": str(appid), "l": "tchinese", "cc": "TW"}, headers=HEADERS, timeout=15)
    g = r.json()[str(appid)]["data"]
    if g.get("type") != "game":
        return None
    # 攤平：list 用 | 連、巢狀 dict 直接往下拿
    return {
        "name": g.get("name"),
        "steam_appid": g.get("steam_appid"),
        "required_age": g.get("required_age"),
        "is_free": g.get("is_free"),
        "header_image": g.get("header_image"),
        "supported_languages": g.get("supported_languages"),
        "developers": "|".join(g.get("developers", [])),
        "publishers": "|".join(g.get("publishers", [])),
        "price_overview.final_formatted": (g.get("price_overview") or {}).get("final"),
        "platforms.windows": (g.get("platforms") or {}).get("windows"),
        "platforms.mac":     (g.get("platforms") or {}).get("mac"),
        "platforms.linux":   (g.get("platforms") or {}).get("linux"),
        "genres.id": "|".join(str(x.get("id")) for x in g.get("genres", [])),
        "genres.description": "|".join((x.get("description") or "") for x in g.get("genres", [])),
        "release_date.date": (g.get("release_date") or {}).get("date"),
    }

# --- 測一顆 ---
appid = 1245620  # 你想換哪顆都行
rec = tiny_game_info(appid)

if rec is None:
    print("這顆不是 game。")
else:
    # 先看輸出
    for k, v in rec.items():
        print(f"{k}: {v}")

    # 想立刻存 CSV，就把下面兩行打開
    # os.makedirs("output", exist_ok=True)
    # pd.DataFrame([rec]).to_csv(f"output/steam_game_{appid}.csv", index=False, encoding="utf-8-sig")


name: 艾爾登法環
steam_appid: 1245620
required_age: 18
is_free: False
header_image: https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1245620/header.jpg?t=1748630546
supported_languages: 英文<strong>*</strong>, 法文, 義大利文, 德文, 西班牙文 - 西班牙, 日文, 韓文, 波蘭文, 葡萄牙文 - 巴西, 俄文, 簡體中文, 西班牙文 - 拉丁美洲, 泰文, 繁體中文, 阿拉伯文<br><strong>*</strong>具備完整音效支援的語言
developers: FromSoftware, Inc.
publishers: FromSoftware, Inc.|Bandai Namco Entertainment
price_overview.final_formatted: 179000
platforms.windows: True
platforms.mac: False
platforms.linux: False
genres.id: 1|3
genres.description: 動作|角色扮演
release_date.date: 2022 年 2 月 24 日


In [17]:
import os
import pandas as pd

def to_games_and_genres(rec: dict):
    # 主表：去掉 genres
    game_row = {k: v for k, v in rec.items() if k != "genres"}
    # 關聯表：一列一個 genre
    gg_rows = []
    for g in rec.get("genres") or []:
        gg_rows.append({
            "steam_appid": rec.get("steam_appid"),
            "genre_id": str(g.get("id")) if g.get("id") is not None else None,
            "genre_desc": g.get("description")
        })
    return game_row, gg_rows

# 測一顆
rec = get_game_info(570)  # 換你要的 appid
if rec is None:
    print("不是 game 或取得失敗")
else:
    game_row, gg_rows = to_games_and_genres(rec)

    os.makedirs("output", exist_ok=True)

    # 1) 遊戲主表（單列）
    pd.DataFrame([game_row]).to_csv("output/steam_games.csv",
                                    index=False, encoding="utf-8-sig")
    # 2) 遊戲-類別關聯表（多列）
    pd.DataFrame(gg_rows).to_csv("output/steam_game_genres.csv",
                                 index=False, encoding="utf-8-sig")

    print("saved:")
    print(" - output/steam_games.csv")
    print(" - output/steam_game_genres.csv")


saved:
 - output/steam_games.csv
 - output/steam_game_genres.csv
